# JS09 — Tugas Lab 1: Benchmark ANNOY vs FAISS vs HNSW

**Praktikum Pembelajaran Mesin — Politeknik Negeri Malang**

| | |
|---|---|
| Nama | TBD |
| NIM | TBD |
| Kelas | TBD |
| Tanggal | TBD |

**Tujuan:** menjalankan percobaan performansi ketiga library ANN pada kombinasi **jumlah data** (1.000 vs 1.000.000), **dimensi** (2D vs 5D), dan **metrik jarak**, lalu mencatat build time, query time, dan recall@10 dalam tabel.

**Prasyarat:** library `annoy`, `faiss`, `hnswlib`, `pandas`, `matplotlib`. Tidak ada dataset eksternal.

> Jalankan di **Google Colab**: `Runtime` ▸ `Run all`. Notebook ini cukup memakai CPU, tidak perlu GPU.


# Tugas Lab 1 — Benchmark ANNOY vs FAISS vs HNSW

Modul: `JS09 › Tugas Lab 1`

**Instruksi modul:**

1. Lakukan percobaan pada **metric distance yang berbeda**, **1.000 vs 1.000.000 data**, dan **2D vs 5D data**, untuk algoritma:
   1. ANNOY
   2. FAISS
   3. HNSW
2. Catat performansinya dalam bentuk tabel.

**Format tabel yang diminta modul:**

| Jumlah Data/Dimensi | ANNOY | FAISS | HNSW |
|---|---|---|---|
| 1000/2D | | | |
| 1000/2D | | | |
| 1000000/5D | | | |
| 1000000/5D | | | |

> **Catatan:** tabel di modul mengulang baris `1000/2D` dan `1000000/5D` — kemungkinan salah ketik. Notebook ini memakai empat kombinasi yang masuk akal: **1.000/2D, 1.000/5D, 1.000.000/2D, 1.000.000/5D**.

**Definisi yang dipakai di notebook ini:**

- **ANNOY** — `euclidean`, 10 tree, `search_k` default (`n_trees × k`).
- **FAISS** — `IndexIVFFlat` (ANN: klaster IVF), `nlist = 100`, `nprobe = 10`. `IndexFlatL2` dipakai terpisah sebagai **exact ground truth**.
- **HNSW** — `hnswlib`, `M = 16`, `ef_construction = 100`, `ef = 50`.
- **Recall@10** — irisan hasil ANN dengan 10 tetangga exact, dibagi 10, dirata-rata atas 100 query.
- **Query time** — total waktu pencarian 100 query dibagi 100 (ms/query).

**Metrik jarak:** grid utama memakai Euclidean. Perbandingan **Euclidean vs Cosine** dijalankan terpisah pada 100.000 vektor (lihat bagian `[tambahan]` di bawah) agar waktu eksekusi tetap wajar. Untuk memakai dua metrik pada seluruh grid, ubah `METRIK_GRID = ['euclidean', 'cosine']` — waktu menjadi ±2× lebih lama.


## Langkah 1 — Konfigurasi Eksperimen

Semua parameter eksperimen dikumpulkan di satu cell agar mudah diubah. Bila RAM atau waktu Colab terbatas, ganti `N_SIZES = [1_000, 1_000_000]` menjadi `[1_000, 100_000]`.

In [ ]:
# [tambahan] Konfigurasi eksperimen
N_SIZES = [1_000, 1_000_000]      # ganti 1_000_000 -> 100_000 bila RAM/waktu terbatas
DIMS = [2, 5]
METRIK_GRID = ['euclidean']       # isi ['euclidean', 'cosine'] untuk grid penuh

K = 10          # jumlah nearest neighbor
NQ = 100        # jumlah query untuk pengukuran waktu & recall
SEED = 42

N_TREES = 10    # Annoy
NLIST = 100     # FAISS IVF: jumlah klaster
NPROBE = 10     # FAISS IVF: klaster yang diperiksa saat query

EF_CONSTRUCTION = 100   # HNSW
M_HNSW = 16             # HNSW
EF_SEARCH = 50          # HNSW

## Langkah 2 — Install & Import Library

In [ ]:
# Install bila belum tersedia (Colab)
try:
    import annoy  # noqa: F401
except ImportError:
    !pip install -q annoy

try:
    import faiss  # noqa: F401
except ImportError:
    !pip install -q faiss-cpu

try:
    import hnswlib  # noqa: F401
except ImportError:
    !pip install -q hnswlib

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from annoy import AnnoyIndex
import faiss
import hnswlib

## Langkah 3 — Fungsi Benchmark

Tiga hal diukur untuk setiap kombinasi: **build time**, **query time per query**, dan **recall@10** terhadap exact NN. Ground truth dihitung dengan FAISS `IndexFlat` (L2 untuk Euclidean, Inner Product untuk Cosine pada vektor yang sudah dinormalisasi).

Untuk metrik **Cosine**, vektor dinormalisasi L2 lebih dulu sehingga *inner product* setara dengan *cosine similarity*. ANNOY memakai `angular`, HNSW memakai `cosine`, FAISS memakai `METRIC_INNER_PRODUCT`.

In [ ]:
def siapkan_vektor(X, metrik):
    """Normalisasi L2 untuk cosine; kembalikan float32 apa adanya untuk euclidean."""
    X = np.asarray(X, dtype='float32')
    if metrik == 'cosine':
        norm = np.linalg.norm(X, axis=1, keepdims=True)
        X = X / np.maximum(norm, 1e-12)
    return X.astype('float32')


def recall_at_k(pred, gt, k=K):
    return float(np.mean([
        len(set(np.asarray(p).ravel()[:k]) & set(np.asarray(g).ravel()[:k])) / k
        for p, g in zip(pred, gt)
    ]))


def bench(n, d, metrik='euclidean'):
    rng = np.random.default_rng(SEED)
    X = rng.random((n, d)).astype('float32')
    Q = rng.random((NQ, d)).astype('float32')

    Xb = siapkan_vektor(X, metrik)
    Qb = siapkan_vektor(Q, metrik)

    # --- pilih metrik tiap library ---
    if metrik == 'euclidean':
        flat, metrik_faiss = faiss.IndexFlatL2(d), faiss.METRIC_L2
        space_annoy, space_hnsw = 'euclidean', 'l2'
    else:  # cosine
        flat, metrik_faiss = faiss.IndexFlatIP(d), faiss.METRIC_INNER_PRODUCT
        space_annoy, space_hnsw = 'angular', 'cosine'

    # --- exact / ground truth ---
    t0 = time.time(); flat.add(Xb); t_build_flat = time.time() - t0
    t0 = time.time(); _, gt = flat.search(Qb, K)
    t_q_flat = (time.time() - t0) * 1000 / len(Qb)

    # --- ANNOY ---
    t0 = time.time()
    ann = AnnoyIndex(d, space_annoy)
    for i, v in enumerate(Xb):
        ann.add_item(i, v)
    ann.build(N_TREES)
    t_build_ann = time.time() - t0
    t0 = time.time()
    pred_ann = [ann.get_nns_by_vector(q, K) for q in Qb]
    t_q_ann = (time.time() - t0) * 1000 / len(Qb)

    # --- FAISS IVF ---
    quantizer = faiss.IndexFlatL2(d) if metrik == 'euclidean' else faiss.IndexFlatIP(d)
    ivf = faiss.IndexIVFFlat(quantizer, d, NLIST, metrik_faiss)
    t0 = time.time(); ivf.train(Xb); ivf.add(Xb); t_build_ivf = time.time() - t0
    ivf.nprobe = NPROBE
    t0 = time.time(); _, pred_ivf = ivf.search(Qb, K)
    t_q_ivf = (time.time() - t0) * 1000 / len(Qb)

    # --- HNSW ---
    t0 = time.time()
    hnsw = hnswlib.Index(space=space_hnsw, dim=d)
    hnsw.init_index(max_elements=n, ef_construction=EF_CONSTRUCTION, M=M_HNSW)
    hnsw.add_items(Xb)
    t_build_hnsw = time.time() - t0
    hnsw.set_ef(EF_SEARCH)
    t0 = time.time(); pred_hnsw, _ = hnsw.knn_query(Qb, k=K)
    t_q_hnsw = (time.time() - t0) * 1000 / len(Qb)

    baris = [
        ('Exact (FAISS Flat)', t_build_flat, t_q_flat, recall_at_k(gt, gt)),
        ('ANNOY', t_build_ann, t_q_ann, recall_at_k(pred_ann, gt)),
        ('FAISS IVF', t_build_ivf, t_q_ivf, recall_at_k(pred_ivf, gt)),
        ('HNSW', t_build_hnsw, t_q_hnsw, recall_at_k(pred_hnsw, gt)),
    ]
    return [{'metrik': metrik,
             'data': f'{n:,}'.replace(',', '.'),
             'dimensi': f'{d}D',
             'algoritma': a,
             'build (s)': b,
             'query (ms/query)': q,
             'recall@10': r} for a, b, q, r in baris]

## Langkah 4 — Jalankan Grid Utama (Euclidean)

> **Perkiraan waktu di Colab CPU:** ±5–8 menit untuk empat kombinasi. Bagian terlama adalah build HNSW pada 1.000.000 vektor (±1–2 menit per kombinasi). Cell akan mencetak progres setiap kombinasi.

In [ ]:
hasil = []
for n in N_SIZES:
    for d in DIMS:
        for metrik in METRIK_GRID:
            print(f"Menjalankan {n:,}".replace(",", "."), f"data / {d}D / {metrik} ...", flush=True)
            hasil += bench(n, d, metrik)

tugas1 = pd.DataFrame(hasil)
print("\nSelesai:", len(tugas1), "baris hasil")
tugas1.round(4)

## Langkah 5 — Tabel Hasil

Tabel pivot di bawah memisahkan tiga metrik yang diminta: **build time**, **query time**, dan **recall@10**.

In [ ]:
# [tambahan] Pivot: recall@10
tabel_recall = tugas1.pivot_table(index=['metrik', 'data', 'dimensi'],
                                  columns='algoritma', values='recall@10')
tabel_recall.round(4)

In [ ]:
# [tambahan] Pivot: query time (ms/query)
tabel_waktu = tugas1.pivot_table(index=['metrik', 'data', 'dimensi'],
                                 columns='algoritma', values='query (ms/query)')
tabel_waktu.round(4)

In [ ]:
# [tambahan] Pivot: build time (detik)
tabel_build = tugas1.pivot_table(index=['metrik', 'data', 'dimensi'],
                                 columns='algoritma', values='build (s)')
tabel_build.round(3)

**Tabel laporan (isi dengan angka dari pivot di atas):**

| Jumlah Data/Dimensi | metrik | ANNOY | FAISS | HNSW |
|---|---|---|---|---|
| 1.000 / 2D | Euclidean | TBD | TBD | TBD |
| 1.000 / 5D | Euclidean | TBD | TBD | TBD |
| 1.000.000 / 2D | Euclidean | TBD | TBD | TBD |
| 1.000.000 / 5D | Euclidean | TBD | TBD | TBD |

*(Metrik yang dicatat: query time ms/query dan recall@10. Build time dilaporkan terpisah pada pivot `tabel_build`.)*

### [tambahan] Perbandingan metrik Euclidean vs Cosine (100.000 vektor)

Modul meminta percobaan pada metrik jarak yang berbeda. Supaya tidak menunggu dua kali lipat pada 1 juta vektor, perbandingan metrik dijalankan pada 100.000 vektor untuk 2D dan 5D.

In [ ]:
# [tambahan] Bandingkan metrik pada 100.000 vektor
N_METRIK = 100_000

hasil_metrik = []
for d in DIMS:
    for metrik in ['euclidean', 'cosine']:
        print(f"Metrik {metrik} / {N_METRIK:,} data / {d}D ...", flush=True)
        hasil_metrik += bench(N_METRIK, d, metrik)

tugas1_metrik = pd.DataFrame(hasil_metrik)
tugas1_metrik.round(4)

In [ ]:
# [tambahan] Pivot perbandingan metrik: recall@10 dan query time
print("=== Recall@10 ===")
display(tugas1_metrik.pivot_table(index=['data', 'dimensi'], columns=['metrik', 'algoritma'],
                                  values='recall@10').round(4))
print("=== Query time (ms/query) ===")
display(tugas1_metrik.pivot_table(index=['data', 'dimensi'], columns=['metrik', 'algoritma'],
                                  values='query (ms/query)').round(4))

### [tambahan] Grafik query time dan rata-rata performansi

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
tabel_waktu.plot(kind='bar', ax=ax, logy=True)
ax.set_ylabel('Query time (ms/query, log)')
ax.set_title('Query time ANN per kombinasi data/dimensi (Euclidean)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# [tambahan] Rata-rata performansi tiap algoritma pada seluruh kombinasi
tugas1.groupby('algoritma')[['build (s)', 'query (ms/query)', 'recall@10']].mean().round(4)

**Analisis:** TBD

*(Jawab dengan angka dari tabel: (1) algoritma dengan query tercepat pada setiap ukuran data, (2) bagaimana kenaikan jumlah data dan dimensi memengaruhi build time dan query time, (3) apakah recall semua algoritma tetap tinggi, dan algoritma mana yang recall-nya paling turun, (4) perbedaan hasil Euclidean vs Cosine pada tabel metrik, (5) library mana yang paling seimbang untuk data besar.)*

---
## Kesimpulan Tugas Lab 1

**Pengaruh jumlah data (1.000 → 1.000.000) terhadap build & query time:** TBD

**Pengaruh dimensi (2D → 5D):** TBD

**Perbandingan recall@10 antar algoritma:** TBD

**Pengaruh metrik Euclidean vs Cosine:** TBD

**Algoritma terbaik untuk data besar menurut percobaan ini:** TBD